In [6]:
import os
import pickle
import torch
from torch.utils.data import Dataset
import random

AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20  # padding 또는 unknown
}
    
class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61, aug=False, symmetry=False):
        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df = df
        self.max_depth = max_depth
        self.win_size = win_size
        self.half_win = win_size // 2  # 중심에서 양쪽 길이
        self.aug = aug
        self.symmetry = symmetry
        
    def __len__(self):
        return len(self.df) * 2 if self.symmetry else len(self.df)

    def __getitem__(self, idx):

        if self.symmetry:
            real_idx = idx // 2
            is_reverse = idx % 2  # 0이면 정방향, 1이면 역방향
        else:
            real_idx = idx
            is_reverse = 0

        row = self.df.iloc[real_idx]
        uid = row["PDB"]
        mut_pos = int(row["POS"]) - 1  # 1-based → 0-based
        mut = row["MT"].upper()
        label = row["DDG"]

        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]  # 보통 첫 줄이 ref seq

        # 변이 반영된 mut_seq 생성
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq):
            mut_seq[mut_pos] = mut

        # --- 핵심 로직: 1, 2행 구성 ---
        if is_reverse == 0:
            # 정방향 (Mut -> WT): 기존 방식
            seqs_to_use = [mut_seq, list(query_seq)]
            target_label = label
        else:
            # 역방향 (WT -> Mut): 대칭 증강
            seqs_to_use = [list(query_seq), mut_seq]
            target_label = -label  # 라벨 부호 반전
            
        if len(msa_seqs) > 1:
            seqs_to_use += [list(seq) for seq in msa_seqs[1:self.max_depth - 2]]

        if self.aug:
            msa_part = seqs_to_use[2:]  # 변이+ref 제외
            random.shuffle(msa_part)   # 순서 섞기
            seqs_to_use = seqs_to_use[:2] + msa_part

        centered_msa = []
        for seq in seqs_to_use:
            window = []
            for i in range(self.win_size):
                seq_idx = mut_pos - self.half_win + i
                if 0 <= seq_idx < len(seq):
                    aa = seq[seq_idx]
                else:
                    aa = '-'
                window.append(AA_TO_INDEX.get(aa, 20))
            centered_msa.append(window)

        # depth padding
        while len(centered_msa) < self.max_depth:
            centered_msa.append([20] * self.win_size)  # 20은 패딩 인덱스

        msa_tensor = torch.tensor(centered_msa[:self.max_depth]).long()  # [D, L]
        msa_tensor = msa_tensor.transpose(0, 1)  # [L, D]

        return {
            "msa": msa_tensor,  # [L, D]
            "label": torch.tensor(target_label).float()
        }


In [7]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba
import torch.nn.functional as F
import numpy as np


# --- Input Embedding ---
class MSAInputEmbedding(nn.Module):
    def __init__(self, vocab_size=21, dim=256):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=dim)

    def forward(self, x):  # x: (B, L, D)
        return self.embedding(x)  # → (B, L, D, C)

# --- Cross-Axial Mamba Block ---
class CrossAxialMambaMSA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm_L = nn.RMSNorm(dim, eps=1e-8)
        self.norm_D = nn.RMSNorm(dim, eps=1e-8)

        self.mamba_L = Mamba(d_model=dim, expand=1)
        self.conv_D = nn.Sequential(
            nn.Conv1d(dim, dim, kernel_size=5, padding=2),
            nn.SiLU()
        )

    def forward(self, x):  # x: (B, L, D, C)
        B, L, D, C = x.shape
        center_L = L // 2  # = 30

        # --- D-axis: only at center L position ---
        x_d_center = self.norm_D(x[:, center_L])  # (B, D, C)
        x_d = x_d_center.transpose(1, 2)          # (B, C, D)
        d_out = self.conv_D(x_d).transpose(1, 2).unsqueeze(1)  # (B, 1, D, C)

        # --- L-axis: full Mamba ---
        x_l = self.norm_L(x).permute(0, 2, 1, 3).contiguous().view(B * D, L, C)
        l_out = self.mamba_L(x_l).view(B, D, L, C).permute(0, 2, 1, 3)  # (B, L, D, C)

        # --- Residual ---
        # d_out is only for center, rest is zero
        d_full = torch.zeros_like(x)
        d_full[:, center_L:center_L+1] = d_out

        return x + d_full + l_out
    
# # --- Encoder ---
class MSAEncoder(nn.Module):
    def __init__(self, num_layers=8, dim=256):
        super().__init__()
        self.embeddings = MSAInputEmbedding(dim=dim)
        self.blocks = nn.ModuleList([
            CrossAxialMambaMSA(dim) for _ in range(num_layers)
        ])
        self.norm_f = nn.RMSNorm(dim, eps=1e-8)

    def forward(self, x):  # x: (B, L, D)
        x = self.embeddings(x)  # → (B, L, D, C)
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x)  # (B, L, D, C)


class SqueezeExcitation3D(nn.Module):
    def __init__(self, in_channels, reduction=24):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.se = nn.Sequential(
            nn.Conv3d(in_channels, in_channels // reduction, kernel_size=1),
            nn.SiLU(),
            nn.Conv3d(in_channels // reduction, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        scale = self.se(self.pool(x))
        return x * scale

class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=6, kernel_size=3, stride=1, se_reduction=16):
        super().__init__()
        mid_ch = in_ch * expand_ratio

        self.use_res_connect = (stride == 1 and in_ch == out_ch)

        self.expand = nn.Sequential(
            nn.Conv3d(in_ch, mid_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        ) if expand_ratio != 1 else nn.Identity()

        self.depthwise = nn.Sequential(
            nn.Conv3d(mid_ch, mid_ch, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size//2, groups=mid_ch, bias=False),
            nn.BatchNorm3d(mid_ch),
            nn.SiLU()
        )

        self.se = SqueezeExcitation3D(mid_ch, reduction=se_reduction)

        self.project = nn.Sequential(
            nn.Conv3d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        identity = x
        out = self.expand(x)
        out = self.depthwise(out)
        out = self.se(out)
        out = self.project(out)

        if self.use_res_connect:
            return out + identity
        else:
            return out

import torch
import torch.nn as nn
import torch.nn.functional as F

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)

class CoordAtt3D(nn.Module):
    def __init__(self, inp, oup, reduction=16):
        super().__init__()
        # 축별 pooling
        self.pool_d = nn.AdaptiveAvgPool3d((None, 1, 1))  # keep D
        self.pool_h = nn.AdaptiveAvgPool3d((1, None, 1))  # keep H
        self.pool_w = nn.AdaptiveAvgPool3d((1, 1, None))  # keep W

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv3d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm3d(mip)
        self.act = h_swish()

        # 축별 복원 conv
        self.conv_d = nn.Conv3d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_h = nn.Conv3d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv3d(mip, oup, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        B, C, D, H, W = x.size()

        # --- 축별 pooling ---
        x_d = self.pool_d(x)  # [B,C,D,1,1]
        x_h = self.pool_h(x)  # [B,C,1,H,1]
        x_w = self.pool_w(x)  # [B,C,1,1,W]

        # 축 정렬을 위해 permute
        x_h = x_h.permute(0, 1, 3, 2, 4)  # [B,C,H,1,1]
        x_w = x_w.permute(0, 1, 4, 2, 3)  # [B,C,W,1,1]

        # concat along "length" dimension
        y = torch.cat([x_d, x_h, x_w], dim=2)  # [B,C,D+H+W,1,1]
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        # 다시 split
        x_d, x_h, x_w = torch.split(y, [D, H, W], dim=2)

        # 축 되돌리기
        x_h = x_h.permute(0, 1, 3, 2, 4)  # [B,C,1,H,1]
        x_w = x_w.permute(0, 1, 3, 4, 2)  # [B,C,1,1,W]

        # attention map
        a_d = self.conv_d(x_d).sigmoid()  # [B,C,D,1,1]
        a_h = self.conv_h(x_h).sigmoid()  # [B,C,1,H,1]
        a_w = self.conv_w(x_w).sigmoid()  # [B,C,1,1,W]

        out = identity * a_d * a_h * a_w
        return out

    
class VoxelBranch(nn.Module):
    def __init__(self, in_ch=63, emb_dim=128):
        super().__init__()
        
        # 3D 구조 백본
        self.backbone = nn.Sequential(
            MBConv3D(in_ch, 32, expand_ratio=6),     # [7×7×7]
            MBConv3D(32, 32, expand_ratio=6),
            MBConv3D(32, 48, expand_ratio=6),
            MBConv3D(48, 48, expand_ratio=6),
            MBConv3D(48, 64, expand_ratio=6),
            MBConv3D(64, 64, expand_ratio=6, stride=2),  # 다운샘플링: → [4×4×4]
            MBConv3D(64, 64, expand_ratio=6),
            MBConv3D(64, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, 96, expand_ratio=6),
            MBConv3D(96, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6),
            MBConv3D(emb_dim, emb_dim, expand_ratio=6)
        )

        self.coordatt = CoordAtt3D(emb_dim, emb_dim)

        self.pool = nn.AdaptiveAvgPool3d(1)  # → [B, 128, 1, 1, 1]

        # Mutation Embedding (64 + 64 → 128)
        self.ref_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_emb = nn.Embedding(21, emb_dim // 2)  # 64
        self.mut_fusion = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU()
        )

        self.refine = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.BatchNorm1d(emb_dim),
            nn.SiLU()
        )

    def forward(self, x, ref_idx, mut_idx):
        x = self.backbone(x)               # [B, 128, 7, 7, 7]
        x = self.coordatt(x) 
        x = self.pool(x).squeeze(-1).squeeze(-1).squeeze(-1)  # → [B, 128]

        # Mutation embedding
        ref_vec = self.ref_emb(ref_idx)    # [B, 64]
        mut_vec = self.mut_emb(mut_idx)    # [B, 64]
        mut_feat = self.mut_fusion(torch.cat([ref_vec, mut_vec], dim=1))  # [B, 128]

        # Combine structure & mutation features
        x = x + mut_feat                   # [B, 128]

        return self.refine(x)       # [B, 128]


class CenterAwarePooling(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)

    def forward(self, x):  # x: (B, L, D, C)
        B, L, D, C = x.shape
        center_L = L // 2

        # Query: center residue [B,D,C] → [B*D,1,C]
        query = x[:, center_L].reshape(B * D, 1, C)

        # Key/Value: 전체 window [B,L,D,C] → [B*D,L,C]
        keyval = x.permute(0, 2, 1, 3).reshape(B * D, L, C)

        # Attention
        attn_out, _ = self.attn(query, keyval, keyval)  # [B*D,1,C]

        # Depth 방향 평균 → [B,C]
        pooled = attn_out.view(B, D, C).mean(dim=1)
        return pooled
    
class MSABranch(nn.Module):
    def __init__(self, num_layers=4, dim=128):
        super().__init__()
        self.encoder = MSAEncoder(num_layers=num_layers, dim=dim)
        self.pooling = CenterAwarePooling(dim, num_heads=4)

        self.refine = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU()
        )

    def forward(self, x):  # x: (B,L,D)
        x = self.encoder(x)        # [B,L,D,C]
        x = self.pooling(x)        # [B,C]  ← center+context 반영
        return self.refine(x)      # [B,C]
    
class EvoStructCLIP(nn.Module):
    def __init__(self, voxel_ch=63, mb_layers=8, embed_dim=128, dropout_p=0.3):
        super().__init__()
        self.voxel_encoder = VoxelBranch(in_ch=voxel_ch, emb_dim=embed_dim)
        self.msa_encoder = MSABranch(num_layers=mb_layers, dim=embed_dim)

        # log(1 / 0.07) ≈ 2.6592 → exp(logit_scale) ≈ 14.2857
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.SiLU(),
            nn.Dropout(dropout_p),
            nn.Linear(embed_dim, 1)
        )

    def _get_vector_norm(self, tensor):
        return F.normalize(tensor, dim=-1, eps=1e-8)

    def forward(self, voxel, ref_idx, mut_idx, msa, inference_only=False):

        if inference_only:
            # msa-only
            msa_feat = self.msa_encoder(msa)   # [B,128]
            logits = self.classifier(msa_feat) # [B,1]
            return {
                "logits": logits,
                "msa_feat": msa_feat
            }
        
        # Raw features
        voxel_feat = self.voxel_encoder(voxel, ref_idx, mut_idx)  # [B, 128]
        msa_feat = self.msa_encoder(msa)                          # [B, 128]

        voxel_embeds = F.normalize(voxel_feat, dim=-1, eps=1e-8)
        msa_embeds   = F.normalize(msa_feat, dim=-1, eps=1e-8)

        logits_per_voxel = torch.matmul(voxel_embeds, msa_embeds.t())
        logits_per_voxel = logits_per_voxel * self.logit_scale.exp().to(voxel_embeds.device)
        logits_per_msa = logits_per_voxel.t() 

        # msa-only classification
        logits = self.classifier(msa_feat)

        return {
            "logits": logits,
            "logits_per_msa": logits_per_msa,
            "logits_per_voxel": logits_per_voxel,
            "voxel_feat": voxel_feat,
            "msa_feat": msa_feat
        }

In [8]:
import torch
import torch.nn as nn

class FeatureAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        # 3개의 특징(Global, Mut, WT) 사이의 관계를 파악하는 Attention
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.SiLU(),
            nn.Linear(dim * 2, dim)
        )
        self.norm_f = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, 3, 128] (3은 Global, Mut, WT)
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm_f(x + ffn_out)
        return x

class StabilityRegressor(nn.Module):
    def __init__(self, clip_model_path, embed_dim=128, unfreeze_all=False):
        super().__init__()
        # 1. 백본 로드
        base_model = EvoStructCLIP(voxel_ch=46, mb_layers=6, embed_dim=embed_dim)
        ckpt = torch.load(clip_model_path, map_location="cpu")
        base_model.load_state_dict(ckpt, strict=True)
        
        self.msa_branch = base_model.msa_encoder 
        
        # --- Freeze 해제 로직 ---
        if unfreeze_all:
            for param in self.msa_branch.parameters():
                param.requires_grad = True
        else:
            # 부분 해제: 마지막 블록과 정제 레이어만 해제 (더 안전함)
            for param in self.msa_branch.parameters():
                param.requires_grad = False
            for param in self.msa_branch.encoder.blocks[-1].parameters():
                param.requires_grad = True
            for param in self.msa_branch.refine.parameters():
                param.requires_grad = True
        
        self.raw_norm = nn.LayerNorm(embed_dim)
        
        # 2. Feature Attention & Regressor (기존과 동일)
        self.feature_attn = FeatureAttentionBlock(embed_dim)
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim * 3, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Linear(128, 1)
        )

    def forward(self, msa):
        # [중요] with torch.no_grad()를 제거하여 Gradient가 흐르게 함
        # (A) Encoder 통과
        x = self.msa_branch.encoder(msa)
        
        B, L, D, C = x.shape
        center_L = L // 2
        query = x[:, center_L].reshape(B * D, 1, C)
        keyval = x.permute(0, 2, 1, 3).reshape(B * D, L, C)
        attn_out, _ = self.msa_branch.pooling.attn(query, keyval, keyval)
        per_seq_feats = attn_out.view(B, D, C)
        
        global_pooled = per_seq_feats.mean(dim=1)
        global_feat = self.msa_branch.refine(global_pooled)
        
        feat0_raw = self.raw_norm(per_seq_feats[:, 0, :])
        feat1_raw = self.raw_norm(per_seq_feats[:, 1, :])
            
        # (B) Feature Attention
        features = torch.stack([global_feat, feat0_raw, feat1_raw], dim=1) 
        features = self.feature_attn(features)
        combined = features.reshape(B, -1)
        
        return self.regressor(combined).squeeze(-1)

In [9]:
import torch.optim as optim
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

def pearson_loss(x, y):
    mx = torch.mean(x)
    my = torch.mean(y)
    xm, ym = x - mx, y - my
    r_num = torch.sum(xm * ym)
    r_den = torch.sqrt(torch.sum(xm**2) * torch.sum(ym**2) + 1e-8)
    r = r_num / r_den
    return 1 - r  # 상관계수가 1에 가까울수록 Loss는 0

def train_one_fold(fold_idx, train_df, val_df, msa_dict_path, clip_weight_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. 데이터로더 구성 (학습 효율을 위해 num_workers 설정)
    train_ds = MSADataset(train_df, msa_dict_path, symmetry=True)
    val_ds = MSADataset(val_df, msa_dict_path)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=8, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)
    
    # 2. 모델 및 최적화 설정
    model = StabilityRegressor(clip_weight_path).to(device)
    criterion = nn.HuberLoss(delta=1.0) # MSE와 MAE의 장점을 합친 Loss
    optimizer = optim.AdamW(model.regressor.parameters(), lr=1e-3, weight_decay=0.05) # 초기 LR 1e-3
    
    # 3. Cosine Annealing Scheduler 추가
    # T_max는 반주기 에폭 수입니다. 보통 전체 에폭 수와 같게 설정하여 끝까지 서서히 줄어들게 합니다.
    epochs = 10000
    scheduler = CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-6)
    
    # 4. Early Stopping 설정
    patience = 500 # 15 에폭 동안 Pearson 점수가 오르지 않으면 중단
    counter = 0
    best_pearson = -1.0
     
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        total_loss = 0
        for batch in train_loader:
            msa = batch["msa"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            preds = model(msa)
            loss = criterion(preds, labels) + 0.5 * pearson_loss(preds, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # --- Validation Phase ---
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                msa = batch["msa"].to(device)
                labels = batch["label"].to(device)
                preds = model(msa)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        # Pearson 상관계수 계산
        fold_pearson, _ = pearsonr(all_labels, all_preds)
        
        # 스케줄러 스텝 (Cosine Decay 적용)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # --- Early Stopping & Model Save Logic ---
        if fold_pearson > best_pearson:
            best_pearson = fold_pearson
            torch.save(model.state_dict(), f"best_reg_fold_{fold_idx}.pth")
            counter = 0 # 개선되었으므로 카운터 리셋
        else:
            counter += 1 # 개선되지 않음
            
        print(f"Fold {fold_idx} | Ep {epoch+1:3d}/{epochs} | LR: {current_lr:.6f} | Loss: {total_loss/len(train_loader):.4f} | Pearson: {fold_pearson:.4f} | Best: {best_pearson:.4f}")
        
        if counter >= patience:
            print(f"!!! Early Stopping triggered at epoch {epoch+1} !!!")
            break

    del model, optimizer, scheduler, train_loader, val_loader
    return best_pearson

In [10]:
import pandas as pd
df = pd.read_csv(r"/mnt/c/Users/Kunny/Documents/GitHub/CAGI/EvoStructCLIP/Stability/S2450.tsv", sep="\t", )

# 경로 설정
msa_dict_path = "/mnt/e/CAGI_data/msa_dict_S2450.pkl"
clip_weight_path = "/mnt/e/CAGI_data/best_model_250918_clip_epoch27.pth"
fold_list = sorted(df['CVFOLD'].unique())

final_results = []

for f_idx in fold_list:
    print(f"\n--- Starting 5-Fold Cross Validation: Fold {f_idx} ---")
    val_df = df[df['CVFOLD'] == f_idx].copy()
    train_df = df[df['CVFOLD'] != f_idx].copy()
    
    best_score = train_one_fold(f_idx, train_df, val_df, msa_dict_path, clip_weight_path)
    final_results.append(best_score)

print("\n" + "="*40)
print(f"5-Fold CV Average Pearson Correlation: {np.mean(final_results):.4f}")
print("="*40)


--- Starting 5-Fold Cross Validation: Fold 0 ---
Fold 0 | Ep   1/10000 | LR: 0.001000 | Loss: 1.1652 | Pearson: 0.4145 | Best: 0.4145
Fold 0 | Ep   2/10000 | LR: 0.001000 | Loss: 1.0996 | Pearson: 0.4141 | Best: 0.4145
Fold 0 | Ep   3/10000 | LR: 0.001000 | Loss: 1.0661 | Pearson: 0.3920 | Best: 0.4145
Fold 0 | Ep   4/10000 | LR: 0.001000 | Loss: 1.0224 | Pearson: 0.4193 | Best: 0.4193
Fold 0 | Ep   5/10000 | LR: 0.001000 | Loss: 1.0236 | Pearson: 0.4378 | Best: 0.4378
Fold 0 | Ep   6/10000 | LR: 0.001000 | Loss: 0.9923 | Pearson: 0.3640 | Best: 0.4378
Fold 0 | Ep   7/10000 | LR: 0.001000 | Loss: 0.9912 | Pearson: 0.4097 | Best: 0.4378
Fold 0 | Ep   8/10000 | LR: 0.001000 | Loss: 0.9789 | Pearson: 0.4116 | Best: 0.4378
Fold 0 | Ep   9/10000 | LR: 0.001000 | Loss: 0.9561 | Pearson: 0.3873 | Best: 0.4378
Fold 0 | Ep  10/10000 | LR: 0.001000 | Loss: 0.9574 | Pearson: 0.3712 | Best: 0.4378
Fold 0 | Ep  11/10000 | LR: 0.001000 | Loss: 0.9477 | Pearson: 0.4149 | Best: 0.4378
Fold 0 | Ep  12

KeyboardInterrupt: 